# Line Emission Denoising — Classical Baselines

**What this answers.** The project's premise is that a learned denoiser beats *traditional
data processing algorithms*. Every comparison run so far has been learned-vs-learned
(V7 / V9 / V12 / beam / sweep) or learned-vs-raw-dirty. **No classical filter had ever been
scored on the line-emission data**, so the question a reviewer will certainly ask — *how much
better than a Gaussian filter?* — had no answer on this dataset. This notebook produces it, on
exactly the protocol the U-Net is judged on.

**Reference to beat** (V12, U-Net): validation PSNR 32.95 dB / SSIM 0.9857, and 5-cube holdout
moment maps **M0 +69.8%±15.2% | M1 +17.5%±7.8% | M2 +20.1%±14.3%**.

**The comparison is deliberately generous to the classical side**, so that a win for the network
is a real win and not a straw man:

1. **Filters run at native 600×600.** The U-Net denoises at 256×256 and is bilinearly resampled
   back, paying a round-trip resolution penalty the filters never pay. §6 quantifies that penalty.
2. **Every filter parameter is tuned, not guessed** — swept on the *validation* cubes using the
   same fixed PSNR metric the hyperparameter sweep scores on, then frozen before the holdout
   evaluation. Tuning on validation and never on holdout is the same discipline the learned
   models follow.
3. Filters are applied to the continuum-subtracted cube in physical units, which is what an
   astronomer would actually do.

**No GPU and no training.** Run this in a **CPU-only** session — it costs zero GPU quota and can
run in parallel with a training notebook.

**Kaggle setup:** Internet on, GPU *off*, `Add Input` → your line-emission Dataset.

## 0. Bootstrap (clone repo for `src/`, locate data)

In [ ]:
import os, sys, subprocess, glob

ON_KAGGLE = os.path.exists('/kaggle')
BRANCH = 'midterm-prep'          # classical baselines live here; switch to line-emission after merge
if ON_KAGGLE:
    REPO='/kaggle/working/EXXA'; PKG=os.path.join(REPO,'DENOISING_DIFFUSION')
    if not os.path.exists(REPO):
        subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',
                        'https://github.com/KrishanYadav333/EXXA.git',REPO], check=True)
    else:
        subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
        subprocess.run(['git','-C',REPO,'reset','--hard','origin/'+BRANCH], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','bettermoments'], check=True)
    os.chdir(os.path.join(PKG,'notebooks'));  sys.path.insert(0, PKG)
    hits = glob.glob('/kaggle/input/**/*_dirty.fits', recursive=True)
    DATA_DIR = os.path.dirname(os.path.dirname(hits[0])) if hits else None
else:
    if os.path.basename(os.getcwd()) != 'notebooks' and os.path.isdir('notebooks'): os.chdir('notebooks')
    sys.path.insert(0, os.path.abspath('..'))
    DATA_DIR = '../data/Line Emission Data'
print('cwd:', os.getcwd(), '| DATA_DIR:', DATA_DIR)

## 0b. Pull latest code (re-run anytime — no kernel restart)

In [ ]:
import os, sys, subprocess
ON_KAGGLE = os.path.exists('/kaggle'); REPO = '/kaggle/working/EXXA'
BRANCH = 'midterm-prep'
if ON_KAGGLE and os.path.exists(REPO):
    subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard','origin/'+BRANCH], check=True)
    print(subprocess.run(['git','-C',REPO,'log','--oneline','-1'],
                         capture_output=True, text=True).stdout.strip())
for _m in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.')]:
    del sys.modules[_m]
print('src.* cleared — re-run the imports cell below.')

## 1. Imports and config (CPU only — no torch device needed)

In [ ]:
import csv, time
import numpy as np
import matplotlib.pyplot as plt

from src.data.cube_split import split_cubes
from src.data.fits_cube_dataset import FITSChannelDataset, continuum_of
from src.evaluation.classical import (DEFAULT_GRIDS, apply_filter, denoise_cube,
                                      summarise_improvements, tune_on_validation)

SEED = 42
np.random.seed(SEED)
TARGET_SIZE        = 256    # only for the channel-level tuning metric (matches the model's view)
N_SAMPLES          = 150
SUBTRACT_CONTINUUM = True
CONTINUUM_N        = 5
OUT_DIR            = '../results'
os.makedirs(OUT_DIR, exist_ok=True)

# V12 reference, for the comparison table
V12 = {'psnr': 32.95, 'ssim': 0.9857, 'mse': 0.000681,
       'M0': (69.8, 15.2), 'M1': (17.5, 7.8), 'M2': (20.1, 14.3)}
print('classical baseline config | continuum n =', CONTINUUM_N, '| grids:', dict(DEFAULT_GRIDS))

## 2. Cube-level split (identical to the U-Net notebooks)

In [ ]:
train_cubes, val_cubes, holdout_cubes = split_cubes(data_dir=DATA_DIR, n_holdout=3,
                                                     val_fraction=0.2, seed=SEED)

## 3. Validation channels for tuning

The same `FITSChannelDataset` the models train on — continuum-subtracted, shared dirty-scale
normalised, 256×256 — so the tuning metric is measured on exactly the data the network sees.
Augmentation stays **off**: this is an evaluation split.

In [ ]:
val_ds = FITSChannelDataset(val_cubes, n_samples=N_SAMPLES, target_size=TARGET_SIZE, seed=SEED,
                            subtract_continuum=SUBTRACT_CONTINUUM, continuum_n=CONTINUUM_N)
print('validation channels available for tuning:', len(val_ds))

## 4. Tune each filter on validation

Selection is by **mean PSNR** — the same fixed metric `run_sweep` scores learned configs on, so
"best classical" and "best learned" are chosen the same way. If an optimum lands on a grid edge
the grid did not bracket it and should be widened; the printout makes that visible.

In [ ]:
t0 = time.time()
tuned = tune_on_validation(val_ds, verbose=True)
print(f'tuning took {time.time()-t0:.0f}s (CPU)')

print('\n' + '=' * 72)
print('{:<12} {:>8} {:>10} {:>9} {:>11}'.format('method', 'param', 'PSNR', 'SSIM', 'MSE'))
print('-' * 72)
for name in ('none', 'gaussian', 'median', 'wiener'):
    r = tuned[name]
    label = 'dirty' if name == 'none' else name
    print('{:<12} {:>8} {:>10.4f} {:>9.4f} {:>11.6f}'.format(
        label, str(r['param']), r['psnr'], r['ssim'], r['mse']))
print('{:<12} {:>8} {:>10.4f} {:>9.4f} {:>11.6f}'.format('U-Net V12', '-', V12['psnr'], V12['ssim'], V12['mse']))
print('=' * 72)

best_classical = max(('gaussian', 'median', 'wiener'), key=lambda m: tuned[m]['psnr'])
print('best classical filter:', best_classical, '| param', tuned[best_classical]['param'],
      '| PSNR {:.4f} dB'.format(tuned[best_classical]['psnr']))
print('U-Net V12 advantage at channel level: {:+.2f} dB'.format(
    V12['psnr'] - tuned[best_classical]['psnr']))

## 5. All-5-holdout moment maps — the decisive comparison

Identical protocol to the U-Net evaluation: denoise every channel of all 5 held-out cubes,
rebuild the cube, and compare `bettermoments` M0/M1/M2 against clean, scoring the improvement
over the dirty baseline as `100 × (1 − |denoised − clean| / |dirty − clean|)`. Same formula, same
cubes, same continuum treatment, so these percentages sit directly beside V12's.

Runs at **native 600×600** — the filters' best case.

In [ ]:
import bettermoments as bm
from astropy.io import fits
from src.evaluation.moment_maps import generate_moment_maps

def mdiff(a, b):
    mask = np.isfinite(a) & np.isfinite(b)
    return float(np.nanmean(np.abs(a[mask] - b[mask])))

def load_csub(path):
    """Continuum-subtracted cube in physical units, plus its FITS header."""
    with fits.open(path, memmap=False) as hdul:
        raw = np.ascontiguousarray(hdul[0].data).astype(np.float32)
        hdr = hdul[0].header.copy()
    return raw - continuum_of(raw, CONTINUUM_N)[None, :, :], hdr

# cache each holdout cube's clean/dirty moment maps once -- they are reused by every method
print('precomputing clean/dirty moment maps for', len(holdout_cubes), 'holdout cubes...')
cache = {}
for ho in holdout_cubes:
    dirty_csub, hdr = load_csub(ho['dirty'])
    clean_csub, _   = load_csub(ho['clean'])
    _, velax = bm.load_cube(ho['dirty'])
    cache[ho['folder']] = {
        'dirty_csub': dirty_csub, 'hdr': hdr, 'velax': velax,
        'clean': generate_moment_maps(None, data_velax=(clean_csub, velax)),
        'dirty': generate_moment_maps(None, data_velax=(dirty_csub, velax)),
    }
    print('  cached', ho['folder'], '| shape', dirty_csub.shape)

METHODS = [('none', None)] + [(m, tuned[m]['param']) for m in ('gaussian', 'median', 'wiener')]
moments = ['M0', 'M1', 'M2']
all_rows, summaries = {}, {}
MAP_CUBE = holdout_cubes[0]['folder']      # cube whose actual maps get displayed in section 5b
moment_maps = {}                           # method -> (M0, M1, M2) for MAP_CUBE

for method, param in METHODS:
    rows = []
    label = 'dirty (unfiltered)' if method == 'none' else f'{method}({param})'
    print(f'\n=== {label} ===', flush=True)
    for ho in holdout_cubes:
        e = cache[ho['folder']]
        den = denoise_cube(e['dirty_csub'], method, param)
        n0, n1, n2 = generate_moment_maps(None, data_velax=(den, e['velax']))
        if ho['folder'] == MAP_CUBE:
            moment_maps[method] = (n0, n1, n2)      # keep the maps, not just the scores
        row = {'cube': ho['folder']}
        for nm, cl, di, no in zip(moments, e['clean'], e['dirty'], (n0, n1, n2)):
            dd, nn = mdiff(cl, di), mdiff(cl, no)
            row['dirty_' + nm] = round(dd, 6)
            row['imp_' + nm] = round(100.0 * (1 - nn / dd), 2) if dd > 0 else float('nan')
        rows.append(row)
        print('  {:<24} M0 {:>7.1f}%  M1 {:>7.1f}%  M2 {:>7.1f}%'.format(
            ho['folder'], row['imp_M0'], row['imp_M1'], row['imp_M2']))
    all_rows[method] = rows
    summaries[method] = summarise_improvements(rows)
    s = summaries[method]
    print('  -> ' + '  '.join('{} {:+.1f}%±{:.1f}'.format(m, s[m]['mean'], s[m]['std']) for m in moments))

## 5b. The moment maps themselves

Percentages compress three 2D scientific products into one number each, and that number can
hide *where* a method fails — a filter can score well on mean absolute difference while
smearing the velocity field, and M1/M2 are the maps that carry the kinematic signal.

Every method's actual M0/M1/M2 maps for one held-out cube, against clean ground truth and the
dirty input. Each moment row shares a colour scale set by the clean map, so panels are directly
comparable across methods rather than each being auto-scaled to flatter itself.

In [ ]:
e = cache[MAP_CUBE]
rows_to_show = [('clean (truth)', e['clean']), ('dirty (input)', e['dirty'])] + \
               [(('dirty (unfiltered)' if m == 'none' else f'{m}({tuned[m]["param"]})'),
                 moment_maps[m]) for m, _ in METHODS if m != 'none']

cmaps = {'M0': 'inferno', 'M1': 'RdBu_r', 'M2': 'viridis'}
fig, axes = plt.subplots(len(rows_to_show), 3, figsize=(11, 3.1 * len(rows_to_show)))

# one colour scale per moment, fixed by the CLEAN map, so nothing is flattered by autoscaling
scales = {}
for j, m in enumerate(moments):
    ref = np.asarray(e['clean'][j], dtype=float)
    finite = ref[np.isfinite(ref)]
    if m == 'M1':                                  # velocity: symmetric about the median
        c = float(np.median(finite)); half = float(np.percentile(np.abs(finite - c), 98))
        scales[m] = (c - half, c + half)
    else:
        scales[m] = (float(np.percentile(finite, 1)), float(np.percentile(finite, 99)))

for i, (label, maps) in enumerate(rows_to_show):
    for j, m in enumerate(moments):
        ax = axes[i, j]
        vmin, vmax = scales[m]
        im = ax.imshow(maps[j], origin='lower', cmap=cmaps[m], vmin=vmin, vmax=vmax)
        ax.set_xticks([]); ax.set_yticks([])
        if i == 0:
            ax.set_title({'M0': 'Moment 0 (intensity)', 'M1': 'Moment 1 (velocity)',
                          'M2': 'Moment 2 (dispersion)'}[m], fontweight='bold')
        if j == 0:
            ax.set_ylabel(label, fontsize=9)
        if i == len(rows_to_show) - 1:
            fig.colorbar(im, ax=axes[:, j].tolist(), fraction=0.02, pad=0.02)

fig.suptitle(f'Moment maps for {MAP_CUBE} — shared colour scale per moment',
             fontweight='bold', y=0.995)
maps_path = os.path.join(OUT_DIR, 'classical_moment_maps.png')
plt.savefig(maps_path, dpi=140, bbox_inches='tight'); plt.show()
print('saved ->', maps_path)
print('\nRead M1 and M2 carefully: a method can win on M0 (total intensity) while visibly')
print('degrading the velocity structure, which is the part that carries the kinematics.')

## 5c. Residual maps — where each method actually errs

`|method − clean|` per moment, on a shared scale. Structured residuals (rings, edges, the
disk rim) mean a method fails systematically on real features; flat noise-like residuals mean
it is only leaving noise behind.

In [ ]:
res_rows = [(('dirty (input)'), e['dirty'])] + \
           [((f'{m}({tuned[m]["param"]})'), moment_maps[m]) for m, _ in METHODS if m != 'none']

fig, axes = plt.subplots(len(res_rows), 3, figsize=(11, 3.1 * len(res_rows)))
res_scale = {}
for j, m in enumerate(moments):
    allres = []
    for _, maps in res_rows:
        d = np.abs(np.asarray(maps[j], float) - np.asarray(e['clean'][j], float))
        allres.append(d[np.isfinite(d)])
    res_scale[m] = (0.0, float(np.percentile(np.concatenate(allres), 99)))

for i, (label, maps) in enumerate(res_rows):
    for j, m in enumerate(moments):
        ax = axes[i, j]
        d = np.abs(np.asarray(maps[j], float) - np.asarray(e['clean'][j], float))
        im = ax.imshow(d, origin='lower', cmap='magma',
                       vmin=res_scale[m][0], vmax=res_scale[m][1])
        ax.set_xticks([]); ax.set_yticks([])
        if i == 0:
            ax.set_title(f'|{m} - clean|', fontweight='bold')
        if j == 0:
            ax.set_ylabel(label, fontsize=9)
        if i == len(res_rows) - 1:
            fig.colorbar(im, ax=axes[:, j].tolist(), fraction=0.02, pad=0.02)

fig.suptitle(f'Absolute moment-map error for {MAP_CUBE} (darker = better)',
             fontweight='bold', y=0.995)
res_path = os.path.join(OUT_DIR, 'classical_moment_residuals.png')
plt.savefig(res_path, dpi=140, bbox_inches='tight'); plt.show()
print('saved ->', res_path)

## 6. Resolution-penalty control — what the 256×256 round trip costs

In [ ]:
# The U-Net denoises at 256 and is resampled back to 600. Running the BEST classical filter
# through the same round trip isolates method from resolution: any gap between this row and the
# native row in section 5 is the penalty the network pays purely for working at 256.
import torch
import torch.nn.functional as F

param = tuned[best_classical]['param']
rows = []
print(f'=== {best_classical}({param}) through the model\'s 256 round trip ===')
for ho in holdout_cubes:
    e = cache[ho['folder']]
    d = e['dirty_csub']
    C, H, W = d.shape
    t = torch.from_numpy(d)[:, None]
    small = F.interpolate(t, (TARGET_SIZE, TARGET_SIZE), mode='bilinear', align_corners=False)[:, 0].numpy()
    filt = denoise_cube(small, best_classical, param)
    back = F.interpolate(torch.from_numpy(filt)[:, None], (H, W),
                         mode='bilinear', align_corners=False)[:, 0].numpy()
    n0, n1, n2 = generate_moment_maps(None, data_velax=(back, e['velax']))
    row = {'cube': ho['folder']}
    for nm, cl, di, no in zip(moments, e['clean'], e['dirty'], (n0, n1, n2)):
        dd, nn = mdiff(cl, di), mdiff(cl, no)
        row['imp_' + nm] = round(100.0 * (1 - nn / dd), 2) if dd > 0 else float('nan')
    rows.append(row)
all_rows[best_classical + '_256rt'] = rows
summaries[best_classical + '_256rt'] = summarise_improvements(rows)
s_rt, s_nat = summaries[best_classical + '_256rt'], summaries[best_classical]
for m in moments:
    print('  {}: native {:+.1f}%  ->  256 round trip {:+.1f}%   (penalty {:+.1f} pp)'.format(
        m, s_nat[m]['mean'], s_rt[m]['mean'], s_rt[m]['mean'] - s_nat[m]['mean']))

## 7. Master comparison table

Everything on one protocol. The learned rows are prior verified results, restated here for
comparison; the classical rows are produced by this notebook.

In [ ]:
LEARNED = [('U-Net V12 (reference)', V12['M0'], V12['M1'], V12['M2']),
           ('U-Net + beam',          (59.8, 33.0), (19.2, 8.4), (23.2, 20.2))]

print('=' * 92)
print('{:<30} {:>18} {:>18} {:>18}'.format('method', 'M0 (%)', 'M1 (%)', 'M2 (%)'))
print('-' * 92)
order = [('none', 'dirty (unfiltered)')] + \
        [(m, '{} sigma/size={}'.format(m, tuned[m]['param'])) for m in ('gaussian', 'median', 'wiener')] + \
        [(best_classical + '_256rt', f'{best_classical} @256 round trip')]
for key, label in order:
    s = summaries[key]
    print('{:<30} {:>10.1f} ±{:<6.1f} {:>10.1f} ±{:<6.1f} {:>10.1f} ±{:<6.1f}'.format(
        label, s['M0']['mean'], s['M0']['std'], s['M1']['mean'], s['M1']['std'],
        s['M2']['mean'], s['M2']['std']))
print('-' * 92)
for label, m0, m1, m2 in LEARNED:
    print('{:<30} {:>10.1f} ±{:<6.1f} {:>10.1f} ±{:<6.1f} {:>10.1f} ±{:<6.1f}'.format(
        label, m0[0], m0[1], m1[0], m1[1], m2[0], m2[1]))
print('=' * 92)

bc = summaries[best_classical]
print('\nHEADLINE: U-Net V12 vs best classical ({} {}):'.format(best_classical, tuned[best_classical]['param']))
for m in moments:
    print('  {}: {:+.1f}% (V12) vs {:+.1f}% (classical)  ->  {:+.1f} pp'.format(
        m, V12[m][0], bc[m]['mean'], V12[m][0] - bc[m]['mean']))
print('\nNOTE: a classical filter that scores near zero on a moment is not "failing" -- it means')
print('the filter neither helps nor hurts that statistic relative to the dirty cube.')

## 8. Figure — moment-map improvement by method

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=False)
plot_keys = [('none', 'dirty'), ('gaussian', 'Gaussian'), ('median', 'median'),
             ('wiener', 'Wiener'), ('V12', 'U-Net V12')]
colors = ['#9aa0a6', '#4E91C7', '#5FA8A0', '#C7A44E', '#E8715A']

for ax, m in zip(axes, moments):
    means = [(V12[m][0] if k == 'V12' else summaries[k][m]['mean']) for k, _ in plot_keys]
    stds  = [(V12[m][1] if k == 'V12' else summaries[k][m]['std'])  for k, _ in plot_keys]
    x = np.arange(len(plot_keys))
    ax.bar(x, means, yerr=stds, capsize=5, color=colors, alpha=0.9,
           error_kw=dict(elinewidth=1.4, ecolor='#333'))
    ax.axhline(0, color='#333', lw=0.8, ls='--')
    ax.set_xticks(x); ax.set_xticklabels([lbl for _, lbl in plot_keys], rotation=30, ha='right')
    ax.set_title({'M0': 'Moment 0 (intensity)', 'M1': 'Moment 1 (velocity)',
                  'M2': 'Moment 2 (dispersion)'}[m])
    ax.set_ylabel('improvement over dirty (%)'); ax.grid(axis='y', alpha=0.3)
fig.suptitle('Learned vs classical denoising — 5 held-out cubes, identical protocol',
             fontweight='bold')
plt.tight_layout()
fig_path = os.path.join(OUT_DIR, 'classical_vs_learned_moments.png')
plt.savefig(fig_path, dpi=140); plt.show()
print('figure saved ->', fig_path)

## 9. Persist CSVs to /kaggle/working

In [ ]:
tune_csv = os.path.join(OUT_DIR, 'classical_tuning_validation.csv')
with open(tune_csv, 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['method', 'param', 'psnr', 'ssim', 'mse'])
    for name, r in tuned.items():
        for p, m in (r['all'] or [(r['param'], r)]):
            w.writerow([name, p, round(m['psnr'], 4), round(m['ssim'], 5), round(m['mse'], 8)])
print('saved ->', tune_csv)

hold_csv = os.path.join(OUT_DIR, 'classical_holdout_moments.csv')
with open(hold_csv, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['method', 'param', 'cube', 'imp_M0', 'imp_M1', 'imp_M2'])
    for method, rows in all_rows.items():
        p = tuned.get(method, {}).get('param', '')
        for r in rows:
            w.writerow([method, p, r['cube'], r['imp_M0'], r['imp_M1'], r['imp_M2']])
        s = summaries[method]
        w.writerow([method, p, 'MEAN'] + [round(s[m]['mean'], 2) for m in moments])
        w.writerow([method, p, 'STD'] + [round(s[m]['std'], 2) for m in moments])
print('saved ->', hold_csv)

import shutil
if ON_KAGGLE:
    for p in (tune_csv, hold_csv, fig_path, maps_path, res_path):
        if os.path.exists(p):
            shutil.copy2(p, '/kaggle/working/' + os.path.basename(p))
            print('persisted ->', '/kaggle/working/' + os.path.basename(p))
    print('\ndownload these from the kernel Output tab and commit them to results/')
else:
    print('not on Kaggle — nothing to persist')